1. 词表初始化 → 2. 文本数据预处理 → 3. 预分词（隔离特殊 Token） → 4. BPE 训练合并子词 → 5. 扩充完整总词表

### 词表初始化以及文本数据预处理

<font size = 4 >
计算机实际底层的词表应该是{0: b'\x00', 1: b'\x01', 2: b'\x02', 3: b'\x03', 4: b'\x04', 5: b'\x05',...}

相当于就是：{ 编号：UTF-8编码后的字节 }，所以我们可以直接用bytes([i])来生成这个词表。依旧体现出扩建词表的必要性。
</font>

In [39]:
# 输出基础词表，print(bytes([256])) 会直接报错，因为bytes()只能接受0~255的整数，超过这个范围就会报错。
# 最终的词表 = 基础词表 + 特殊Token + 自定义词表（这里暂时不考虑） + 扩展的词表
# 可以先给这些词表划分区域，防止键值冲突。也可以等BPE扩展词表结束后再加入特殊Token词表和自定义词表，最后再做一次去重（主要是自定义词表可能会冲突）。
# 其实只需要基础的跟扩展的就可以了，具体还是要看任务，来选择加不加词表，这里就只描述必备的基础词表跟扩展词表，并且将特殊token词表加到最后。
final_vocab = {}
diy_vocab = {} # 自定义词表
special_vocab = {        
        9900: b"<|endoftext|>",
        9901: b"<|startoftext|>",
        9902: b"<|pad|>",
        9903: b"<|unk|>",
        9904: b"<|mask|>"} # 特殊Token词表


def init_vocab():
    vocab = {i:bytes([i]) for i in range(256)}
    final_vocab.update(vocab) # 将初始词表加入总词表中
    final_vocab.update(diy_vocab) # 将自定义词表加入总词表中
    final_vocab.update(special_vocab) # 将特殊Token词表加入总词表中
    return final_vocab

# print(init_vocab())

# 隔离特殊Token，假设文本是 Hello<|endoftext|>World，如果简单地把 <|endoftext|> 删除，左右两段会变成 HelloWorld，BPE 就可能学习跨文档边界的错误合并。
# <|endoftext|>表示文本的结束，<|startoftext|>表示文本的开始，<|pad|>表示填充，<|unk|>表示未知词，<|mask|>表示掩码。
import re

special_token_strs = [
    "<|endoftext|>",
    "<|startoftext|>",
    "<|pad|>",
    "<|unk|>",
    "<|mask|>"
]

def split_special_text(text, special_token_strs):
    # 无特殊token，直接返回原文本
    if not special_token_strs:
        return [text]
    else:
        # 1. re.escape(t)：给每个特殊符号里的特殊字符加转义符，防止正则解析出错
        #    例：<|endoftext|> 里面的 | 在正则里有特殊含义，转义后变成 \<\|endoftext\|>
        # 2. "|".join()：用 | 拼接所有转义后的标记，正则里 | 代表“或”（便于匹配）
        # 拼接结果示例：\<\|endoftext\|\>|\<\|startoftext\|\>|\<\|pad\|\>...
        # 整条正则含义：匹配文本里任意一个特殊标记
        special_token_pattern = "|".join(re.escape(x) for x in special_token_strs)
        print(special_token_pattern) # \<\|endoftext\|\>|\<\|startoftext\|\>|\<\|pad\|\>|\<\|unk\|\>|\<\|mask\|\>
        
        # re.split：按照正则匹配到的内容分割文本
        # 重点：正则外面包了()捕获组，分割时不会直接删掉特殊标记，会把标记也放进结果列表
        # 举例：text="大模型<|endoftext|>心脏病"
        # 不加()分割结果：['大模型','心脏病']（标记直接消失）
        # 加()分割结果：['大模型', '<|endoftext|>', '心脏病']（文字和标记都保留）
        split_text = re.split(f"({special_token_pattern})", text)
        print(split_text) # ['大模型', '<|endoftext|>', '心脏病']
      
        # 列表推导式，循环遍历上面分割出来的所有片段parts，做两层筛选：
        # 1. for p in parts：遍历所有分割出来的片段
        # 2. if p not in special_token_strs and p.strip()：过滤掉特殊标记和空白片段（不是特殊标记，并且去掉空格后不是空字符串）
        # 3. p.strip()：去掉保留下来的片段中的多余空格
        # 接上例parts = ['大模型', '<|endoftext|>', '心脏病']
        # 过滤后得到：['大模型','心脏病']
        splited_segment = [f.strip() for f in split_text if f not in special_token_strs and f.strip()]
        
        # 返回清洗完毕、不含任何特殊标记的纯文本片段列表，专门用于BPE训练
        return splited_segment
    
text = "大模型很好<|endoftext|>心脏病很常见<|mask|>深度学习"

splited_segment = split_special_text(text, special_token_strs) # splited_segment: ['大模型很好', '心脏病很常见', '深度学习']
print(splited_segment) # ['大模型很好', '心脏病很常见', '深度学习']

<\|endoftext\|>|<\|startoftext\|>|<\|pad\|>|<\|unk\|>|<\|mask\|>
['大模型很好', '<|endoftext|>', '心脏病很常见', '<|mask|>', '深度学习']
['大模型很好', '心脏病很常见', '深度学习']


### 预分词并统计词频
<font size = 4>
将每一个片段的内容先预先进行一次分词，防止BPE在合并相邻字符的时候出现跨词等错误，相当于先把边界划分出来。
</font>

In [ ]:
# 设置预分词正则表达式，由于使用的是外部的正则分词方式，所以要导入regex包
import regex as re
from collections import Counter

# 主要匹配五类内容：英文缩写、字母串、数字串、标点符号串和空白。所以用中文做实验的话效果可能不会很好
Gpt2_pattern = re.compile(r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

# 统计：预分词单元的字节Token序列在整个预料中的出现次数
raw_counts = Counter()

for segment in ['LLM很好，应该学', 'Hello World!']:
    # print(f"'{segment}' -> {segment.encode('utf-8')}")
    # print(f"'{segment}' -> {list(segment.encode('utf-8'))}") # 将字符串转为字节列表
    # 'LLM很好，应该学' -> b'LLM\xe5\xbe\x88\xe5\xa5\xbd\xef\xbc\x8c\xe5\xba\x94\xe8\xaf\xa5\xe5\xad\xa6'
    # 'LLM很好，应该学' -> [76, 76, 77, 229, 190, 136, 229, 165, 189, 239, 188, 140, 229, 186, 148, 232, 175, 165, 229, 173, 166]
    # 'Hello World!' -> b'Hello World!'
    # 'Hello World!' -> [72, 101, 108, 108, 111, 32, 87, 111, 114, 108, 100, 33]

    # print(Gpt2_pattern.findall(segment)) # ['LLM很好', '，', '应该学'] ['Hello', ' World', '!']
    # 后续BPE就是在预分词的基础上，进一步提取连续的字节对（2-gram）来构建词表。
    # ['LLM很好', '，', '应该学']就是在'LLM很好'中进行迭代

    # step1：预分词
    re_seg = Gpt2_pattern.findall(segment) # 预分词结果
    # print(re_seg)

    # step2：统计每个预分词单元在整个预料中的出现次数(目前还没有转成utf-8的描述方式)
    raw_counts.update(re_seg) 

# print("raw_counts:", raw_counts)
    
# step3：将预分词结果,按字符拆分，并且转成对应的utf-8字节序列
# 将每个不同的预分词单元转换成单字节 Token 序列，同时保留它原来的出现频率

word_list = [] # 保存每个预分词单元对应的字节Token序列
counts_list = [] # 保存每个预分词单元对应的出现次数，就是对应上面的raw_counts字典里的value值

for word, count in raw_counts.items():
    # BPE是以字节为单位的，所以要将每个预分词单元转换成对应的字节序列。
    # BPE后面需要执行字节拼接，例如：b'H' + b'e' -> b'He'，所以全部都需要转成字节序列，不能用数字表示了
    byte_tokens = [
        bytes([b]) for b in word.encode("utf-8")
    ] # 将每个预分词单元转换成对应的字节序列
    word_list.append(byte_tokens)
    counts_list.append(count)

# seg_byte = [ 
#     # [bytes(b) for b in s.encode("utf-8")] # 错误写法 ['Hello', ' World', '!'] -> 字节Token： 字节Token： [[b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00.........]]
#     [bytes([b]) for b in s.encode("utf-8")] # ['Hello', ' World', '!'] -> 字节Token： [[b'H', b'e', b'l', b'l', b'o'], [b' ', b'W', b'o', b'r', b'l', b'd'], [b'!']]
#     for s in re_seg
# ]



word_list: [[b'L', b'L', b'M', b'\xe5', b'\xbe', b'\x88', b'\xe5', b'\xa5', b'\xbd'], [b'\xef', b'\xbc', b'\x8c'], [b'\xe5', b'\xba', b'\x94', b'\xe8', b'\xaf', b'\xa5', b'\xe5', b'\xad', b'\xa6'], [b'H', b'e', b'l', b'l', b'o'], [b' ', b'W', b'o', b'r', b'l', b'd'], [b'!']]
counts_list: [1, 1, 1, 1, 1, 1]


### BPE算法

| 分词方式 | 示例                               | 优点                    | 问题                    |
| ---- | -------------------------------- | --------------------- | --------------------- |
| 单词级  | `Hello World` → `[Hello, World]` | 序列较短                  | 词表巨大，容易出现未登录词         |
| 字符级  | `Hello` → `[H,e,l,l,o]`          | 不容易出现未登录词             | 序列较长，难以复用常见词片段        |
| 字节级  | `很好` → `[E5,BE,88,E5,A5,BD]`     | 只需256个基础Token，可表示任何文本 | 中文和emoji会被拆成很多字节，序列更长 |
 
<font size = 4>
BPE的作用就是在“词表大小”和“序列长度”之间取得平衡：

先保留256个基础字节，保证任何文本都能表示；再根据语料频率，把经常相邻出现的Token逐步合并成更大的Token。

因此，低频或从未见过的内容仍然可以退回到基础字节，高频字符、词片段或单词则可以用更少的 Token 表示。
<font>
<font size = 2>

例如：
[E5,BE,88,E5,A5,BD] 需要6个Token表示

学习为：
(E5,BE)       → E5BE
(E5BE,88)     → 很
(E5,A5)       → E5A5
(E5A5,BD)     → 好
(很,好)       → 很好

最终：[E5,BE,88,E5,A5,BD] → [很好] 仅需一个TOken表示
<font>

##### BPE算法原理
<font size = 4>
(1)BPE的目的是在词表大小和序列长度之间取得平衡。仅使用字节虽然能表示任意文本，但会产生较长的Token序列；直接使用完整单词又会导致词表过大，并产生未登录词。因此，Byte-level BPE先以256个字节作为基础词表，再把语料中频繁相邻的Token逐步合并成更大的Token。

(2)训练时，先对文本进行预分词，以限制BPE只能在每个预分词单元内部合并；随后将预分词单元转换为UTF-8字节Token，并结合其语料频率统计相邻Token pair。每轮选择频率最高的pair，在所有预分词单元中进行合并，同时产生一个新Token并扩充词表。由于合并会改变Token之间的相邻关系，因此需要重新统计并重复这一过程，直到达到目标词表大小或不存在可合并的pair。
<font>

In [ ]:
# 这边我举一个新的例子来演示BPE
# 预分词后的统计结果
from collections import Counter

raw_counts = Counter({
    "low": 5,
    "lower": 2,
    "lowest": 4,
    "slow": 3,
    "slower": 2,
})

# 转成字节Token序列以供BPE使用
words_list = []
counts_list = []

for word, count in raw_counts.items():
    byte_tokens = [
        bytes([b]) for b in word.encode("utf-8")
    ]
    words_list.append(byte_tokens)
    counts_list.append(count)

# 由于构建词表是根据全部的语料进行的，所以针对于某一个2-gram对，需要统计在全部预料中的出现次数
total_freq = Counter()
# 开始进行每一个单元内的BPE迭代
for idx, word in enumerate(words_list):
    # 取出该字符对应的出现次数
    count = counts_list[idx]

    # 便利每个预分词单元的字节Token序列，提取连续的字节对（2-gram）
    for i in range(len(word) - 1):
        # 取出连续的两个字节Token，组成一个字节对
        pair = (word[i], word[i + 1])
        # print(pair) # (b'l', b'o') (b'o', b'w') (b'l', b'o') (b'o', b'w') (b'e', b'r') (b'l', b'o') (b'o', b'w') (b'e', b's') (b's', b't') (b's', b'l') (b'l', b'o') (b'o', b'w') (b'e', b'r')
        # 统计该字节对在全部预料中的出现次数
        # total_freq.update([pair]) # total_freq.update(pair)少了"[]"会导致pair又被单独拆开统计
        # 这里会有个问题，就是只统计了每个预分词单元，但是该单元在整体预料中出现的次数没有加进去
        # 例如：pair: (b'l', b'o')在"low"中出现了5次，所以应该是total_freq.update([pair] * count)才对
        # total_freq.update([pair] * count) 
        # 会有性能问题，[pair] * count 是用「重复count相同数据」加一万次，在没有求和之前都是需要一直保存住的；
        # pair = (b'l', b'o')* 3 输出：[(b'l', b'o'), (b'l', b'o'), (b'l', b'o')] -> 最后再一起统计
        total_freq[pair] += count;
# print("total_freq:", total_freq)

# 频率统计结束后就该找到这一次遍历中频率最高的组合了
best_pair = max(
     total_freq.items(),
     key = lambda item : (item[1], item[0])
)[0] # ((b'o', b'w'), 16)->(b'o', b'w')
# print(best_pair)

# 找到频率最高的组合以后，把 best_pair 拼接产生的 new_token 放进词表。
# 这里我直接进行词表的扩展了，不再额外保留一个扩展词表了
final_vocab = {
    i:bytes([i]) for i in range(256)
} # 初始化最基础的词表
# print(final_vocab)
# 更新词表，加入新token
new_token = best_pair[0] + best_pair[1] # b'ow',如果是数字就没有办法这样相加了。所以才需要用utf-8编码
final_vocab[len(final_vocab)] = new_token
# print(final_vocab[256])  # b'ow'

# 新Token已经加入词表，但当前words_list中仍然保存的是旧划分：
#
# low -> [b'l', b'o', b'w']
#
# 为了让下一轮BPE能够在b'ow'的基础上继续合并，
# 需要把所有预分词单元中的best_pair替换成new_token。
# print(word_list)

for word in words_list:
    # 得到每一个预分词单元
    while(i < len(word)-1) :
        if (word[i] == best_pair[0] and word[i+1] == best_pair[1]):
            word[i:i+2] = [new_token]
            # word[i] = new_token
            # del word[i+1]
            i += 1
        else:
            i += 1
    

print(words_list)

[[b'l', b'o', b'w'], [b'l', b'o', b'w', b'e', b'r'], [b'l', b'o', b'w', b'e', b's', b't'], [b's', b'l', b'o', b'w'], [b's', b'l', b'o', b'w', b'e', b'r']]


#### BPE算法结束条件
（1）正常停止：词表达到目标大小；
（2）提前停止：已经不存在相邻pair；


In [154]:
# 判断目前词表大小是否达到目标大小
target_vocab_size = 9999
len_special_token = 5
if (len(final_vocab) < target_vocab_size - 256 - len_special_token): print("词表未满，继续BPE！")
else: print("词表已满，停止BPE！")

# 判断一下每一个预分词单元内的token序列长度是否为1，为1就表示不存在相邻的，没必要继续合并了
if (all( len(word) == 1 for word in words_list )): print("无相邻pair，停止BPE!")

词表未满，继续BPE！
